## 1. Setting up Kaggle API

In [16]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "kiva_loans.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "kiva/data-science-for-good-kiva-crowdfunding",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

print("First 5 records:", df.head())

/var/folders/cp/wz7fj_wn5nj9ypx5k7l1dxyr0000gn/T/ipykernel_20393/2549967359.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


First 5 records:        id  funded_amount  loan_amount             activity          sector  \
0  653051          300.0        300.0  Fruits & Vegetables            Food   
1  653053          575.0        575.0             Rickshaw  Transportation   
2  653068          150.0        150.0       Transportation  Transportation   
3  653063          200.0        200.0           Embroidery            Arts   
4  653084          400.0        400.0           Milk Sales            Food   

                                                 use country_code   country  \
0            To buy seasonal, fresh fruits to sell.            PK  Pakistan   
1  to repair and maintain the auto rickshaw used ...           PK  Pakistan   
2  To repair their old cycle-van and buy another ...           IN     India   
3  to purchase an embroidery machine and a variet...           PK  Pakistan   
4                           to purchase one buffalo.           PK  Pakistan   

         region currency  partner_id   

## 2. Understanding the dataset

In [21]:
df.shape # (rows, columns)

(671205, 25)

In [22]:
df.info() # Lists all columns, their non-null counts and data types

<class 'pandas.DataFrame'>
RangeIndex: 671205 entries, 0 to 671204
Data columns (total 25 columns):
 #   Column              Non-Null Count   Dtype              
---  ------              --------------   -----              
 0   id                  671205 non-null  int64              
 1   funded_amount       671205 non-null  float64            
 2   loan_amount         671205 non-null  float64            
 3   activity            671205 non-null  str                
 4   sector              671205 non-null  str                
 5   use                 666973 non-null  str                
 6   country_code        671197 non-null  str                
 7   country             671205 non-null  str                
 8   region              614405 non-null  str                
 9   currency            671205 non-null  str                
 10  partner_id          657698 non-null  float64            
 11  posted_time         671205 non-null  datetime64[us, UTC]
 12  disbursed_time      668809 

In [23]:
df["borrower_genders"].value_counts().head(10) # Gender column values and counts

borrower_genders
female                                                                            426502
male                                                                              134710
female, female                                                                     12164
female, female, female                                                             11676
female, female, female, female                                                      9052
female, female, female, female, female                                              7568
female, female, female, female, female, female                                      2674
female, female, female, female, female, female, female, female, female, female      2185
female, female, female, female, female, female, female                              2127
female, female, female, female, female, female, female, female                      1936
Name: count, dtype: int64

In [24]:
df.describe() # Simple statistics (e.g., min, max, avg) of quantitative columns

,id,funded_amount,loan_amount,partner_id,term_in_months,lender_count,days_to_fund,n_female,n_male,num_borrowers
count,6.712050e+05,671205.000000,671205.000000,657698.000000,671205.000000,671205.000000,622874.000000,671205.000000,671205.000000,671205.000000
mean,9.932486e+05,785.995061,842.397107,178.199616,13.739022,20.590922,14.641528,1.596097,0.409568,2.005664
std,1.966113e+05,1130.398941,1198.660073,94.247581,8.598919,28.459551,14.399107,3.034134,1.116451,3.406619
min,6.530470e+05,0.000000,25.000000,9.000000,1.000000,0.000000,-17.505278,0.000000,0.000000,0.000000
25%,8.230720e+05,250.000000,275.000000,126.000000,8.000000,7.000000,5.174265,1.000000,0.000000,1.000000
50%,9.927800e+05,450.000000,500.000000,145.000000,13.000000,13.000000,9.590122,1.000000,0.000000,1.000000
75%,1.163653e+06,900.000000,1000.000000,204.000000,14.000000,24.000000,22.538657,1.000000,1.000000,1.000000
max,1.340339e+06,100000.000000,100000.000000,536.000000,158.000000,2986.000000,420.573264,50.000000,44.000000,50.000000


## 3. About the data

- 426,502 single-woman loans vs. 134,710 single-man (about 64% vs. 20% of all loans).
- Groups go up to 10+ borrowers.
- 48,331 loans (~7%) were never fully funded, so they have no funding time.
- **Note:** Kiva borrowers are mostly women, and group loans are almost all female. Be careful of misinterpreting group-size effects as gender effects.

## 4. Research Question #1: Which variables correlate most with funding time?

### Setting up features: `days_to_fund`, `n_female`, `n_male`, and `gender_group`

In [28]:
import numpy as np
import pandas as pd

# 1. Funding time in days
df["posted_time"] = pd.to_datetime(df["posted_time"])
df["funded_time"] = pd.to_datetime(df["funded_time"])
df["days_to_fund"] = (df["funded_time"] - df["posted_time"]).dt.total_seconds() / 86400

# 2. Count women and men in each loan
genders = df["borrower_genders"].fillna("")
df["n_female"] = genders.str.count(r"\bfemale\b")
df["n_male"] = genders.str.count(r"\bmale\b")
df["num_borrowers"] = df["n_female"] + df["n_male"]

# 3. Label each loan as Female, Male, or Mixed
df["gender_group"] = np.select(
    [df["n_male"] == 0, df["n_female"] == 0],
    ["Female", "Male"],
    default="Mixed",
)

### Checking if features were created properly

In [25]:
df["days_to_fund"].describe()
# Note: A negative min means loan(s) have a `funded_time` before its `posted_time`. Possibly timestamp errors or reposted loans.

count    622874.000000
mean         14.641528
std          14.399107
min         -17.505278
25%           5.174265
50%           9.590122
75%          22.538657
max         420.573264
Name: days_to_fund, dtype: float64

In [20]:
df["gender_group"].value_counts()

gender_group
Female    492302
Male      138523
Mixed      40380
Name: count, dtype: int64

### Cleaning up dataset
Dropped negative or never-funded loans and loans with no gender listed

In [29]:
funded = df[(df["days_to_fund"] > 0) & (df["num_borrowers"] > 0)].copy()
print(f"Kept {len(funded):,} of {len(df):,} loans")
funded["gender_group"].value_counts()

Kept 619,334 of 671,205 loans


gender_group
Female    465160
Male      116937
Mixed      37237
Name: count, dtype: int64